# ☀️ Solar Panel Segmentation & Temporal Analysis (2021–2025)
### End-to-End Google Colab Notebook using **YOLO11m-seg**

This notebook provides a complete workflow:
1. **Environment Setup & Verification** (GPU acceleration).
2. **Direct Dataset Ingestion** from GitHub repository `Hinarashid220/FYP`.
3. **Multi-Year `data.yaml` Generation** across all splits (2021 to 2025).
4. **Training YOLO11m-seg** with satellite-optimized hyper-parameters & augmentations.
5. **Validation & Test Set Evaluation** (mAP50, mAP50-95, PR Curves).
6. **Temporal Growth & Adoption Analysis** (2021–2025 installation growth charts).
7. **Export & Save Best Model Artifacts**.

--- 
## 1. Setup Environment & Dependencies

In [ ]:
# 1. Verify GPU assignment
!nvidia-smi

# 2. Install Ultralytics and dependencies
!pip install -q ultralytics matplotlib opencv-python pyyaml pandas seaborn

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

--- 
## 2. Clone Dataset from GitHub
Clones your GitHub repository `https://github.com/Hinarashid220/FYP.git` directly.

In [ ]:
import os
from pathlib import Path

# Clone the repository if not already present
if not os.path.exists("FYP"):
    print("Cloning dataset from GitHub...")
    !git clone https://github.com/Hinarashid220/FYP.git
else:
    print("Repository FYP is already cloned.")

DATASET_ROOT = Path("FYP/YOLO")
YEARS = ["2021", "2022", "2023", "2024", "2025"]

# Audit available years
found_years = [y for y in YEARS if (DATASET_ROOT / y).exists()]
print(f"Successfully verified {len(found_years)} years of data: {found_years}")

--- 
## 3. Generate Colab-Compatible `data.yaml`
Aggregates train, val, and test splits across all five years (2021–2025).

In [ ]:
import yaml

base_dir = os.path.abspath("FYP/YOLO")

train_dirs = [os.path.join(base_dir, y, "train", "images") for y in YEARS if os.path.exists(os.path.join(base_dir, y, "train", "images"))]
val_dirs   = [os.path.join(base_dir, y, "valid", "images") for y in YEARS if os.path.exists(os.path.join(base_dir, y, "valid", "images"))]
test_dirs  = [os.path.join(base_dir, y, "test", "images")  for y in YEARS if os.path.exists(os.path.join(base_dir, y, "test", "images"))]

config = {
    "path": base_dir,
    "train": train_dirs,
    "val": val_dirs,
    "test": test_dirs,
    "names": {
        0: "solar_panel"
    }
}

config_path = os.path.join(base_dir, "solar_data.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config written to: {config_path}")
print(f"  Train folders: {len(train_dirs)}")
print(f"  Val folders:   {len(val_dirs)}")
print(f"  Test folders:  {len(test_dirs)}")

--- 
## 4. Train YOLO11m-seg Model
Fine-tuned for satellite imagery with:
- **Rotational invariance** (degrees=180)
- **Multi-scale augmentation** (mosaic, mixup, scale)
- **Photometric augmentation** (HSV variation for atmospheric/lighting differences)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 medium instance segmentation model
model = YOLO("yolo11m-seg.pt")

# Train model
results = model.train(
    data="FYP/YOLO/solar_data.yaml",
    epochs=150,           # Full training run
    imgsz=432,            # Native patch resolution
    batch=16,             # Optimal batch size for Colab T4/V100/A100
    patience=30,          # Early stopping
    workers=2,
    device=0,
    # --- Satellite Augmentations ---
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=180,          # 360 degree arbitrary orientation
    flipud=0.5,
    fliplr=0.5,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    # --- Logging & Checkpointing ---
    project="runs/solar-seg",
    name="yolo11m-seg",
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

print("=" * 60)
print("Training Complete!")
print("Best Weights: runs/solar-seg/yolo11m-seg/weights/best.pt")
print("=" * 60)

--- 
## 5. Evaluate Best Model on Unseen Test Split

In [ ]:
best_weights = "runs/solar-seg/yolo11m-seg/weights/best.pt"
val_model = YOLO(best_weights)

# Validate specifically on the test split
metrics = val_model.val(
    data="FYP/YOLO/solar_data.yaml",
    split="test",
    imgsz=432,
    batch=16,
    plots=True,
    project="runs/solar-seg",
    name="yolo11m-seg-test",
)

print("\n" + "=" * 60)
print("YOLO11m-seg Final Test Performance:")
print(f"  Segmentation Mask mAP50:    {metrics.seg.map50:.4f}")
print(f"  Segmentation Mask mAP50-95: {metrics.seg.map:.4f}")
print(f"  Bounding Box mAP50:         {metrics.box.map50:.4f}")
print(f"  Bounding Box mAP50-95:      {metrics.box.map:.4f}")
print("=" * 60)

--- 
## 6. Visualize Predictions on Test Images

In [ ]:
import glob
import matplotlib.pyplot as plt

# Display test predictions batch image
pred_plots = glob.glob("runs/solar-seg/yolo11m-seg-test/val_batch*_pred.jpg")
if not pred_plots:
    pred_plots = glob.glob("runs/solar-seg/yolo11m-seg/val_batch*_pred.jpg")

if pred_plots:
    fig, ax = plt.subplots(figsize=(16, 12))
    img = plt.imread(pred_plots[0])
    ax.imshow(img)
    ax.set_title("YOLO11m-seg: Predictions on Unseen Test Batch", fontsize=16, fontweight="bold")
    ax.axis("off")
    plt.show()
else:
    print("No batch prediction images found yet.")

--- 
## 7. Temporal Growth & Solar Adoption Analysis (2021–2025)
Runs inference on each year to measure:
- Total solar panels detected year-over-year
- Adoption rate (% of regions with panels)
- Per-base installation growth heatmap

In [ ]:
from collections import defaultdict
import json
import numpy as np

model = YOLO(best_weights)
output_dir = Path("runs/solar-seg/temporal-analysis")
output_dir.mkdir(parents=True, exist_ok=True)

yearly_stats = {}
base_stats = defaultdict(lambda: defaultdict(dict))

print("Analyzing temporal solar growth...")
for year in YEARS:
    panel_counts = []
    total_area = 0
    images_with_panels = 0
    total_images = 0

    for split in ["train", "valid", "test"]:
        img_dir = DATASET_ROOT / year / split / "images"
        if not img_dir.exists():
            continue

        results = model.predict(source=str(img_dir), imgsz=432, conf=0.25, save=False, verbose=False)
        for r in results:
            total_images += 1
            n_panels = len(r.masks.data) if r.masks is not None else 0
            panel_counts.append(n_panels)
            if n_panels > 0:
                images_with_panels += 1
                total_area += r.masks.data.sum().item()
                fname = Path(r.path).stem
                parts = fname.split("_")
                for p in parts:
                    if p.startswith("Base"):
                        base_stats[p][year][parts[0]] = n_panels
                        break

    yearly_stats[year] = {
        "total_images": total_images,
        "images_with_panels": images_with_panels,
        "total_panels": sum(panel_counts),
        "avg_panels": np.mean(panel_counts) if panel_counts else 0,
        "adoption_rate": (images_with_panels / total_images * 100) if total_images else 0
    }

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Solar Panel Temporal Growth (2021–2025)", fontsize=18, fontweight="bold")
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]

# 1. Total Panels
ax = axes[0, 0]
counts = [yearly_stats[y]["total_panels"] for y in YEARS]
bars = ax.bar(YEARS, counts, color=colors, edgecolor="white", linewidth=1.5)
ax.set_title("Total Panels Detected", fontweight="bold")
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 5, str(c), ha="center", va="bottom", fontweight="bold")

# 2. Adoption Rate
ax = axes[0, 1]
rates = [yearly_stats[y]["adoption_rate"] for y in YEARS]
ax.plot(YEARS, rates, "o-", color="#E91E63", linewidth=2.5, markersize=8)
ax.fill_between(YEARS, rates, alpha=0.15, color="#E91E63")
ax.set_title("Solar Adoption Rate (%)", fontweight="bold")
ax.set_ylim(0, 100)

# 3. Avg Panels
ax = axes[1, 0]
avgs = [yearly_stats[y]["avg_panels"] for y in YEARS]
ax.plot(YEARS, avgs, "s-", color="#4CAF50", linewidth=2.5, markersize=8)
ax.fill_between(YEARS, avgs, alpha=0.15, color="#4CAF50")
ax.set_title("Average Panels per Image", fontweight="bold")

# 4. Heatmap by Base Location
ax = axes[1, 1]
base_names = sorted(base_stats.keys(), key=lambda x: int(x.replace("Base", "")) if x.replace("Base", "").isdigit() else 0)
heatmap_data = [[sum(base_stats[b].get(y, {}).values()) for y in YEARS] for b in base_names]
if heatmap_data:
    im = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(YEARS)))
    ax.set_xticklabels(YEARS)
    ax.set_yticks(range(len(base_names)))
    ax.set_yticklabels(base_names, fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Panels per Base Location", fontweight="bold")

plt.tight_layout()
chart_file = output_dir / "solar_temporal_growth.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()

# Save statistics JSON
(output_dir / "yearly_stats.json").write_text(json.dumps(yearly_stats, indent=2))
print(f"Growth charts saved to: {chart_file}")

--- 
## 8. Export All Results
Zips trained weights (`best.pt`, `last.pt`), PR curves, confusion matrices, and the temporal growth charts for download.

In [ ]:
# Create zip archive of all training & temporal results
!zip -r yolo11m_solar_complete_results.zip runs/solar-seg
print("\nCreated archive: yolo11m_solar_complete_results.zip")

# Download to local computer (optional in Colab)
# from google.colab import files
# files.download('yolo11m_solar_complete_results.zip')